# Deepfake Live Detector — Colab + Tunnel

This notebook runs the FastAPI backend **for free** using Google Colab's GPU and exposes it publicly via a cloudflare tunnel.

**How it works:**
1. Clone the repo + install deps (torch already in Colab)
2. Start the FastAPI server on port 8000
3. Create a public `*.trycloudflare.com` URL

**Limitations to be aware of:**
- URL changes every time you restart (give the new one before each demo)
- Session dies after ~12h idle — re-run cells to restart
- cloudflared adds a one-click interstitial on first visit
- Not always-on — this is a **demo/portfolio** setup

## Step 1 — Clone repo + install dependencies

In [ ]:
# Download cloudflared (the tunnel binary)
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared -s && \
chmod +x /usr/local/bin/cloudflared
print('cloudflared ready')

In [ ]:
# Clone the repo (includes .pth weights in git)
import os
if not os.path.exists('/content/DeepFake'):
    !git clone https://github.com/Mayurroro/DeepFake.git /content/DeepFake
else:
    !cd /content/DeepFake && git pull
os.chdir('/content/DeepFake')
print('repo ready at', os.getcwd())

In [ ]:
# System libs python-magic + exiftool need (apt, not pip)
!apt-get update -qq && apt-get install -y -qq libmagic1 libimage-exiftool-perl > /dev/null

# Python deps (torch already in Colab — pip skips reinstalling it)
!pip install -q --no-cache-dir \
  fastapi uvicorn[standard] python-multipart pydantic \
  opencv-python-headless Pillow albumentations \
  timm scipy numpy librosa soundfile sounddevice \
  scikit-learn onnxruntime python-magic PyExifTool

## Step 2 — Start the API server (port 8000)

In [ ]:
import subprocess, time, re, urllib.request

# Kill any leftover server/tunnel
subprocess.run('pkill -f uvicorn || true', shell=True, capture_output=True)
subprocess.run('pkill -f cloudflared || true', shell=True, capture_output=True)
time.sleep(1)

# Start uvicorn (background)
server_log = open('/tmp/server.log', 'w')
server = subprocess.Popen(
    ['uvicorn', 'api.fastapi_server:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=server_log, stderr=subprocess.STDOUT
)
print(f'uvicorn started (pid {server.pid})')

# Wait until server is ready
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/', timeout=2)
        print('server is up on :8000')
        break
    except:
        time.sleep(1)
else:
    print('server did not come up — check /tmp/server.log')

## Step 3 — Create public tunnel + get URL

In [ ]:
# Start cloudflared tunnel (background)
tunnel_log = open('/tmp/tunnel.log', 'w')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=tunnel_log, stderr=subprocess.STDOUT
)
print(f'cloudflared started (pid {tunnel.pid})')

# Read the public URL from logs
time.sleep(8)
with open('/tmp/tunnel.log') as f:
    content = f.read()
urls = re.findall(r'https://[\w-]+\.trycloudflare\.com', content)

if urls:
    public_url = urls[-1]
    print(f'\n✅  YOUR PUBLIC URL (give this to anyone):')
    print(f'    {public_url}')
    print(f'\n    Homepage:  {public_url}/')
    print(f'    API docs:  {public_url}/docs')
else:
    print('could not find tunnel URL — check /tmp/tunnel.log')
    !cat /tmp/tunnel.log | tail -30

---
**Portfolio sharing tip:** Run this notebook, copy the URL, and add it to your portfolio/resume. Before each interview, open the notebook, re-run all cells, and share the fresh URL. Total setup time: ~60 seconds.